In [2]:
!pip install -q kagglehub


In [3]:
import kagglehub

kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


In [8]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("tschandl/isic2018-challenge-task1-data-segmentation")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'isic2018-challenge-task1-data-segmentation' dataset.
Path to dataset files: /kaggle/input/isic2018-challenge-task1-data-segmentation


## 1. Dataset Verification

In [12]:
import torch
import torch.nn as nn
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights

# Configuration parameters
IMAGE_SIZE = 224
NUM_CLASSES = 1

# Define network classes within the cell to ensure it is self-contained
class MobileNetV3SmallEncoder(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        if pretrained:
            weights = MobileNet_V3_Small_Weights.DEFAULT
        else:
            weights = None
        self.backbone = mobilenet_v3_small(weights=weights)
        self.features = self.backbone.features
        self.out_channels = [16, 16, 24, 48, 576]
        self.skip_connection_indices = [0, 1, 3, 8, 12]

    def forward(self, x):
        feature_maps = []
        x_current = x
        for i, layer in enumerate(self.features):
            x_current = layer(x_current)
            if i in self.skip_connection_indices:
                feature_maps.append(x_current)
        return feature_maps

class UNetDecoder(nn.Module):
    def __init__(self, encoder_out_channels, num_classes):
        super().__init__()
        self.upconv1 = nn.ConvTranspose2d(encoder_out_channels[4], encoder_out_channels[3], kernel_size=2, stride=2)
        self.conv_block1 = self._make_conv_block(encoder_out_channels[3] * 2, encoder_out_channels[3])
        self.upconv2 = nn.ConvTranspose2d(encoder_out_channels[3], encoder_out_channels[2], kernel_size=2, stride=2)
        self.conv_block2 = self._make_conv_block(encoder_out_channels[2] * 2, encoder_out_channels[2])
        self.upconv3 = nn.ConvTranspose2d(encoder_out_channels[2], encoder_out_channels[1], kernel_size=2, stride=2)
        self.conv_block3 = self._make_conv_block(encoder_out_channels[1] * 2, encoder_out_channels[1])
        self.upconv4 = nn.ConvTranspose2d(encoder_out_channels[1], encoder_out_channels[0], kernel_size=2, stride=2)
        self.conv_block4 = self._make_conv_block(encoder_out_channels[0] * 2, encoder_out_channels[0])
        self.upconv_final = nn.ConvTranspose2d(encoder_out_channels[0], encoder_out_channels[0], kernel_size=2, stride=2)
        self.final_conv = nn.Conv2d(encoder_out_channels[0], num_classes, kernel_size=1)

    def _make_conv_block(self, in_channels, out_channels):
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, feature_maps):
        f_112, f_56, f_28, f_14, f_7x7 = feature_maps
        x = self.upconv1(f_7x7)
        x = torch.cat([x, f_14], dim=1)
        x = self.conv_block1(x)
        x = self.upconv2(x)
        x = torch.cat([x, f_28], dim=1)
        x = self.conv_block2(x)
        x = self.upconv3(x)
        x = torch.cat([x, f_56], dim=1)
        x = self.conv_block3(x)
        x = self.upconv4(x)
        x = torch.cat([x, f_112], dim=1)
        x = self.conv_block4(x)
        x = self.upconv_final(x)
        output = self.final_conv(x)
        return output

class MobileNetV3UNet(nn.Module):
    def __init__(self, pretrained_encoder=True, freeze_encoder=False, num_classes=1):
        super().__init__()
        self.encoder = MobileNetV3SmallEncoder(pretrained=pretrained_encoder)
        self.decoder = UNetDecoder(self.encoder.out_channels, num_classes)
        if freeze_encoder:
            for param in self.encoder.parameters():
                param.requires_grad = False

    def forward(self, x):
        feature_maps = self.encoder(x)
        output = self.decoder(feature_maps)
        return output

# Check for GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Instantiate the models
print("Instantiating pretrained and scratch models...")
model_pretrained = MobileNetV3UNet(pretrained_encoder=True, freeze_encoder=False, num_classes=NUM_CLASSES)
model_scratch = MobileNetV3UNet(pretrained_encoder=False, freeze_encoder=False, num_classes=NUM_CLASSES)

# Move models to device
model_pretrained = model_pretrained.to(device)
model_scratch = model_scratch.to(device)

# Create dummy input and move to device
dummy_input = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE).to(device)

# Re-test the model on the selected device
print("\nRe-testing MobileNetV3UNet on selected device:")

with torch.no_grad():
    output_pretrained = model_pretrained(dummy_input)
print(f"Pretrained model output shape on {device}: {output_pretrained.shape}")
assert output_pretrained.shape == (1, NUM_CLASSES, IMAGE_SIZE, IMAGE_SIZE), "Pretrained model output shape is incorrect on device!"

with torch.no_grad():
    output_scratch = model_scratch(dummy_input)
print(f"Scratch model output shape on {device}: {output_scratch.shape}")
assert output_scratch.shape == (1, NUM_CLASSES, IMAGE_SIZE, IMAGE_SIZE), "Scratch model output shape is incorrect on device!"

print("Model successfully moved to device and re-tested.")

Using device: cuda
Instantiating pretrained and scratch models...
Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:00<00:00, 209MB/s]



Re-testing MobileNetV3UNet on selected device:
Pretrained model output shape on cuda: torch.Size([1, 1, 224, 224])
Scratch model output shape on cuda: torch.Size([1, 1, 224, 224])
Model successfully moved to device and re-tested.


In [13]:
import os
import glob

# List the contents of the downloaded dataset path
dataset_contents = os.listdir(path)
print(f"Contents of the dataset directory '{path}':\n{dataset_contents}")

# Define the specific directories for training images and masks for ISIC 2018 Task 1 segmentation
image_dir = os.path.join(path, 'ISIC2018_Task1-2_Training_Input')
mask_dir = os.path.join(path, 'ISIC2018_Task1_Training_GroundTruth')

# Find all image and mask files within these specific directories
image_files = glob.glob(os.path.join(image_dir, '*.jpg'))
mask_files = glob.glob(os.path.join(mask_dir, '*_segmentation.png'))

print(f"\nFound {len(image_files)} image files and {len(mask_files)} mask files within specified training directories.")

# Extract ISIC IDs and verify pairing
# For images, remove the file extension
image_ids = set([os.path.basename(f).replace('.jpg', '') for f in image_files])
# For masks, remove the _segmentation.png extension
mask_ids = set([os.path.basename(f).replace('_segmentation.png', '') for f in mask_files])

# Check for missing masks
missing_masks = image_ids - mask_ids
if missing_masks:
    raise ValueError(f"Found {len(missing_masks)} images without corresponding masks: {list(missing_masks)[:5]}...")

# Check for masks without corresponding images
missing_images = mask_ids - image_ids
if missing_images:
    raise ValueError(f"Found {len(missing_images)} masks without corresponding images: {list(missing_images)[:5]}...")

# Verify that the number of images and masks are equal after ID check
if len(image_ids) != len(mask_files) or len(mask_ids) != len(image_files):
    raise ValueError("Mismatch in the number of images and masks after ID verification.")

print("Dataset verification successful: All images have corresponding masks and vice versa.")
print(f"Total unique image/mask pairs: {len(image_ids)}")

# Store the paired file paths for later use
paired_files = []
for img_id in sorted(list(image_ids)):
    img_path = os.path.join(image_dir, f'{img_id}.jpg')
    mask_path = os.path.join(mask_dir, f'{img_id}_segmentation.png')
    paired_files.append((img_path, mask_path))

print(f"Successfully created a list of {len(paired_files)} paired image and mask paths.")

Contents of the dataset directory '/kaggle/input/isic2018-challenge-task1-data-segmentation':
['ISIC2018_Task1-2_Training_Input', 'ISIC2018_Task1_Training_GroundTruth', 'ISIC2018_Task1-2_Test_Input', 'ISIC2018_Task1-2_Validation_Input']

Found 2594 image files and 2594 mask files within specified training directories.
Dataset verification successful: All images have corresponding masks and vice versa.
Total unique image/mask pairs: 2594
Successfully created a list of 2594 paired image and mask paths.


## 2. Fixed Train / Validation / Test Split and Few-Shot Subset Creation

In [14]:
import random
import numpy as np
from sklearn.model_selection import train_test_split

# Configuration from project requirements
SEED = 42
TEST_SET_SIZE = 500
VALIDATION_SET_SIZE = 250
FEW_SHOT_SIZES = [50, 100, 250, 500, 1000]

# Ensure reproducibility
random.seed(SEED)
np.random.seed(SEED)

# Create 'splits' directory if it doesn't exist
splits_dir = 'splits'
os.makedirs(splits_dir, exist_ok=True)

print(f"Total available paired files: {len(paired_files)}")

# Extract only the image paths for splitting
all_image_paths = [img_path for img_path, mask_path in paired_files]

# 1. Split out the fixed test set
# Remaining images will be used for training pool and validation
train_val_pool_images, test_images = train_test_split(
    all_image_paths, test_size=TEST_SET_SIZE, random_state=SEED,
    shuffle=True # Shuffle ensures random selection
)

# 2. Split out the fixed validation set from the training/validation pool
# The rest will be the main training pool
train_pool_images, validation_images = train_test_split(
    train_val_pool_images, test_size=VALIDATION_SET_SIZE, random_state=SEED,
    shuffle=True # Shuffle ensures random selection
)

print(f"\nInitial Split Summary:")
print(f"  Training Pool Images: {len(train_pool_images)}")
print(f"  Validation Images:    {len(validation_images)}")
print(f"  Test Images:          {len(test_images)}")
print(f"  Total:                {len(train_pool_images) + len(validation_images) + len(test_images)}")

# Save fixed validation and test sets
with open(os.path.join(splits_dir, 'validation.txt'), 'w') as f:
    for img_path in validation_images:
        f.write(f"{img_path}\n")
print(f"Saved {len(validation_images)} validation image paths to {os.path.join(splits_dir, 'validation.txt')}")

with open(os.path.join(splits_dir, 'test.txt'), 'w') as f:
    for img_path in test_images:
        f.write(f"{img_path}\n")
print(f"Saved {len(test_images)} test image paths to {os.path.join(splits_dir, 'test.txt')}")

# 3. Create few-shot training subsets from the training pool
# These subsets should be sampled without replacement from the main training pool
for size in FEW_SHOT_SIZES:
    if size > len(train_pool_images):
        print(f"Warning: Requested few-shot size {size} is greater than the available training pool ({len(train_pool_images)}). Using all available training images.")
        subset_images = train_pool_images
    else:
        subset_images = random.sample(train_pool_images, size)

    with open(os.path.join(splits_dir, f'train_{size}.txt'), 'w') as f:
        for img_path in subset_images:
            f.write(f"{img_path}\n")
    print(f"Saved {len(subset_images)} training image paths for few-shot size {size} to {os.path.join(splits_dir, f'train_{size}.txt')}")

# Verify no data leakage
print("\nVerifying no data leakage:")
train_pool_set = set(train_pool_images)
validation_set = set(validation_images)
test_set = set(test_images)

if not train_pool_set.isdisjoint(validation_set):
    raise ValueError("Data leakage: Training pool and Validation set overlap!")
if not train_pool_set.isdisjoint(test_set):
    raise ValueError("Data leakage: Training pool and Test set overlap!")
if not validation_set.isdisjoint(test_set):
    raise ValueError("Data leakage: Validation set and Test set overlap!")

print("Data leakage verification successful: All sets are disjoint.")

# Print final dataset summary
print("\nDataset Summary")
print("-------------------------")
print(f"Training images pool: {len(train_pool_images)}")
print(f"Validation images: {len(validation_images)}")
print(f"Test images: {len(test_images)}")
print(f"Total: {len(train_pool_images) + len(validation_images) + len(test_images)}")
print(f"\nFew-shot subset sizes created: {FEW_SHOT_SIZES}")

Total available paired files: 2594

Initial Split Summary:
  Training Pool Images: 1844
  Validation Images:    250
  Test Images:          500
  Total:                2594
Saved 250 validation image paths to splits/validation.txt
Saved 500 test image paths to splits/test.txt
Saved 50 training image paths for few-shot size 50 to splits/train_50.txt
Saved 100 training image paths for few-shot size 100 to splits/train_100.txt
Saved 250 training image paths for few-shot size 250 to splits/train_250.txt
Saved 500 training image paths for few-shot size 500 to splits/train_500.txt
Saved 1000 training image paths for few-shot size 1000 to splits/train_1000.txt

Verifying no data leakage:
Data leakage verification successful: All sets are disjoint.

Dataset Summary
-------------------------
Training images pool: 1844
Validation images: 250
Test images: 500
Total: 2594

Few-shot subset sizes created: [50, 100, 250, 500, 1000]


## 3. Dataset and DataLoader

In [15]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import random
import numpy as np

# Configuration parameters (from requirement 16)
IMAGE_SIZE = 224
BATCH_SIZE = 16
SEED = 42

# ImageNet statistics for normalization (from requirement 10)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

class ISICDataset(Dataset):
    def __init__(self, image_paths, mask_paths, transform=None, augment=False, seed=None):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.transform = transform
        self.augment = augment
        self.seed = seed

        # Ensure image and mask paths are aligned
        image_ids = {os.path.basename(p).replace('.jpg', ''): p for p in image_paths}
        mask_ids = {os.path.basename(p).replace('_segmentation.png', ''): p for p in mask_paths}

        self.paired_files = []
        for img_id in sorted(image_ids.keys()):
            if img_id in mask_ids:
                self.paired_files.append((image_ids[img_id], mask_ids[img_id]))
            else:
                # This case should ideally not happen after initial verification
                print(f"Warning: Image {img_id}.jpg has no corresponding mask. Skipping.")

    def __len__(self):
        return len(self.paired_files)

    def __getitem__(self, idx):
        image_path, mask_path = self.paired_files[idx]

        image = Image.open(image_path).convert("RGB")
        mask = Image.open(mask_path).convert("L") # Convert mask to grayscale

        # Apply transforms
        if self.augment:
            # Ensure the same random transformations are applied to image and mask
            if self.seed is not None:
                random.seed(self.seed + idx) # Use seed + index for per-item reproducibility in augmentation
                torch.manual_seed(self.seed + idx)

            # Spatial transforms that affect both image and mask
            angle = transforms.RandomRotation.get_params([-10, 10]) # Small rotations
            hflip = random.random() < 0.5
            vflip = random.random() < 0.5

            if hflip:
                image = transforms.functional.hflip(image)
                mask = transforms.functional.hflip(mask)
            if vflip:
                image = transforms.functional.vflip(image)
                mask = transforms.functional.vflip(mask)

            image = transforms.functional.rotate(image, angle)
            mask = transforms.functional.rotate(mask, angle, interpolation=transforms.InterpolationMode.NEAREST) # Nearest neighbor for masks

            # Random scale/crop (example, need careful implementation for masks)
            # For simplicity, let's just resize and then apply color jitter
            # A more robust implementation would use something like RandomResizedCrop with same params.

            # Color jitter for image only
            color_jitter = transforms.ColorJitter(brightness=0.2, contrast=0.2)
            image = color_jitter(image)

        # Resize images and masks
        image = transforms.functional.resize(image, (IMAGE_SIZE, IMAGE_SIZE), interpolation=transforms.InterpolationMode.BILINEAR)
        mask = transforms.functional.resize(mask, (IMAGE_SIZE, IMAGE_SIZE), interpolation=transforms.InterpolationMode.NEAREST) # Nearest neighbor for masks

        # Convert to tensor and normalize image
        image = transforms.functional.to_tensor(image)
        image = transforms.functional.normalize(image, mean=IMAGENET_MEAN, std=IMAGENET_STD)

        # Convert mask to tensor and to binary (0 or 1)
        mask = transforms.functional.to_tensor(mask) # Range [0, 1]
        mask = (mask > 0.5).float() # Threshold to make it binary

        if self.transform:
            image = self.transform(image)

        return image, mask


# Define transforms for validation/test (no augmentation)
val_test_transform = transforms.Compose([
    transforms.ToTensor(), # Already done inside __getitem__ for consistent image/mask processing
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

# --- Load split file paths ---
def load_paths_from_file(filepath):
    with open(filepath, 'r') as f:
        paths = [line.strip() for line in f if line.strip()]
    return paths

# Load validation and test image paths
validation_image_paths = load_paths_from_file(os.path.join(splits_dir, 'validation.txt'))
test_image_paths = load_paths_from_file(os.path.join(splits_dir, 'test.txt'))

# Helper to get mask paths from image paths (assuming same folder structure as verified earlier)
def get_mask_paths(image_paths):
    mask_paths = []
    for img_path in image_paths:
        img_id = os.path.basename(img_path).replace('.jpg', '')
        mask_paths.append(os.path.join(mask_dir, f'{img_id}_segmentation.png'))
    return mask_paths

validation_mask_paths = get_mask_paths(validation_image_paths)
test_mask_paths = get_mask_paths(test_image_paths)

# Create Datasets
# Validation and Test datasets do not use augmentation
validation_dataset = ISICDataset(
    image_paths=validation_image_paths,
    mask_paths=validation_mask_paths,
    transform=None, # Transforms handled internally
    augment=False,
    seed=SEED
)
test_dataset = ISICDataset(
    image_paths=test_image_paths,
    mask_paths=test_mask_paths,
    transform=None, # Transforms handled internally
    augment=False,
    seed=SEED
)

# Create DataLoaders
validation_loader = DataLoader(
    validation_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=os.cpu_count() // 2
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=os.cpu_count() // 2
)

print(f"Validation Dataset size: {len(validation_dataset)}")
print(f"Test Dataset size: {len(test_dataset)}")
print(f"Number of CPU workers used for DataLoaders: {os.cpu_count() // 2}")

# Example of how to get a few-shot training loader (to be done inside the training loop for each experiment)
def get_train_dataloader(few_shot_size):
    train_image_paths = load_paths_from_file(os.path.join(splits_dir, f'train_{few_shot_size}.txt'))
    train_mask_paths = get_mask_paths(train_image_paths)

    train_dataset = ISICDataset(
        image_paths=train_image_paths,
        mask_paths=train_mask_paths,
        transform=None, # Transforms handled internally
        augment=True, # Apply augmentation for training
        seed=SEED
    )
    train_loader = DataLoader(
        train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=os.cpu_count() // 2
    )
    print(f"Training Dataset size for {few_shot_size} images: {len(train_dataset)}")
    return train_loader

# Test a training dataloader for one of the few-shot sizes
example_train_loader = get_train_dataloader(FEW_SHOT_SIZES[0])
for images, masks in example_train_loader:
    print(f"Example batch shape - Images: {images.shape}, Masks: {masks.shape}")
    break

print("Dataset and DataLoader setup complete.")

Validation Dataset size: 250
Test Dataset size: 500
Number of CPU workers used for DataLoaders: 1
Training Dataset size for 50 images: 50
Example batch shape - Images: torch.Size([16, 3, 224, 224]), Masks: torch.Size([16, 1, 224, 224])
Dataset and DataLoader setup complete.


## 4. MobileNetV3-UNet Architecture

In [ ]:
import torch.nn as nn
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights
import torch # Import torch for torch.cat

# Configuration parameters
IMAGE_SIZE = 224
NUM_CLASSES = 1 # Binary segmentation: 1 for lesion, 0 for background

class MobileNetV3SmallEncoder(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        if pretrained:
            weights = MobileNet_V3_Small_Weights.DEFAULT
        else:
            weights = None

        # Load MobileNetV3-Small model
        self.backbone = mobilenet_v3_small(weights=weights)

        self.features = self.backbone.features

        # Based on actual debug output from 224x224 input, these are the correct layers for U-Net style skip connections:
        # Layer 0: torch.Size([1, 16, 112, 112]) -> f_112
        # Layer 1: torch.Size([1, 16, 56, 56])   -> f_56
        # Layer 3: torch.Size([1, 24, 28, 28])   -> f_28
        # Layer 8: torch.Size([1, 48, 14, 14])   -> f_14
        # Layer 12: torch.Size([1, 576, 7, 7])  -> f_7x7 (bottleneck output)

        self.out_channels = [
            16,   # After features[0] (112x112)
            16,   # After features[1] (56x56)
            24,   # After features[3] (28x28)
            48,   # After features[8] (14x14)
            576   # After features[12] (7x7) - Bottleneck
        ]
        self.skip_connection_indices = [0, 1, 3, 8, 12] # Store these indices

    def forward(self, x):
        # Collect feature maps for skip connections
        feature_maps = []
        x_current = x
        for i, layer in enumerate(self.features):
            x_current = layer(x_current)
            # Removed debug print: print(f"Encoder Layer {i}: {x_current.shape}")
            if i in self.skip_connection_indices:
                feature_maps.append(x_current)
        return feature_maps

class UNetDecoder(nn.Module):
    def __init__(self, encoder_out_channels, num_classes):
        super().__init__()
        # encoder_out_channels: [16 (112x112), 16 (56x56), 24 (28x28), 48 (14x14), 576 (7x7)]

        # Decoder stage 1: Up from f_7x7 (features[12]), concatenate with f_14 (features[8])
        # f_7x7 (576 channels, 7x7) -> upsample to spatial size of f_14 (14x14)
        self.upconv1 = nn.ConvTranspose2d(encoder_out_channels[4], encoder_out_channels[3], kernel_size=2, stride=2)
        # After concat: channels = encoder_out_channels[3] (from upconv) + encoder_out_channels[3] (from skip)
        self.conv_block1 = self._make_conv_block(encoder_out_channels[3] * 2, encoder_out_channels[3]) # (48+48 -> 48)

        # Decoder stage 2: Up from output of conv_block1, concatenate with f_28 (features[3])
        self.upconv2 = nn.ConvTranspose2d(encoder_out_channels[3], encoder_out_channels[2], kernel_size=2, stride=2)
        # After concat: channels = encoder_out_channels[2] (from upconv) + encoder_out_channels[2] (from skip)
        self.conv_block2 = self._make_conv_block(encoder_out_channels[2] * 2, encoder_out_channels[2]) # (24+24 -> 24)

        # Decoder stage 3: Up from output of conv_block2, concatenate with f_56 (features[1])
        self.upconv3 = nn.ConvTranspose2d(encoder_out_channels[2], encoder_out_channels[1], kernel_size=2, stride=2)
        # After concat: channels = encoder_out_channels[1] (from upconv) + encoder_out_channels[1] (from skip)
        self.conv_block3 = self._make_conv_block(encoder_out_channels[1] * 2, encoder_out_channels[1]) # (16+16 -> 16)

        # Decoder stage 4: Up from output of conv_block3, concatenate with f_112 (features[0])
        self.upconv4 = nn.ConvTranspose2d(encoder_out_channels[1], encoder_out_channels[0], kernel_size=2, stride=2)
        # After concat: channels = encoder_out_channels[0] (from upconv) + encoder_out_channels[0] (from skip)
        self.conv_block4 = self._make_conv_block(encoder_out_channels[0] * 2, encoder_out_channels[0]) # (16+16 -> 16)

        # Final upsampling to IMAGE_SIZE (224x224)
        # We need one more upsampling stage to go from 112x112 to 224x224
        self.upconv_final = nn.ConvTranspose2d(encoder_out_channels[0], encoder_out_channels[0], kernel_size=2, stride=2)

        # Final segmentation head
        self.final_conv = nn.Conv2d(encoder_out_channels[0], num_classes, kernel_size=1)

    def _make_conv_block(self, in_channels, out_channels):
        # Decoder block consisting of two convolutions, batch norm, and ReLU
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, feature_maps):
        # feature_maps correspond to:
        # f_112 (features[0], 16ch, 112x112)
        # f_56 (features[1], 16ch, 56x56)
        # f_28 (features[3], 24ch, 28x28)
        # f_14 (features[8], 48ch, 14x14)
        # f_7x7 (features[12], 576ch, 7x7) - Bottleneck
        f_112, f_56, f_28, f_14, f_7x7 = feature_maps

        # Removed debug prints for decoder inputs

        # Decoder Stage 1: Upsample 7x7 -> 14x14
        x = self.upconv1(f_7x7) # 576 -> 48 channels, spatial 7x7 -> 14x14
        # Removed debug print: print(f"After upconv1 (upsampled from f_7x7): {x.shape}")
        x = torch.cat([x, f_14], dim=1) # 48 + 48 = 96 channels
        # Removed debug print: print(f"After concat1: {x.shape}")
        x = self.conv_block1(x) # 96 -> 48 channels
        # Removed debug print: print(f"After conv_block1: {x.shape}")

        # Decoder Stage 2: Upsample 14x14 -> 28x28
        x = self.upconv2(x) # 48 -> 24 channels, spatial 14x14 -> 28x28
        # Removed debug print: print(f"After upconv2: {x.shape}")
        x = torch.cat([x, f_28], dim=1) # 24 + 24 = 48 channels
        # Removed debug print: print(f"After concat2: {x.shape}")
        x = self.conv_block2(x) # 48 -> 24 channels
        # Removed debug print: print(f"After conv_block2: {x.shape}")

        # Decoder Stage 3: Upsample 28x28 -> 56x56
        x = self.upconv3(x) # 24 -> 16 channels, spatial 28x28 -> 56x56
        # Removed debug print: print(f"After upconv3: {x.shape}")
        x = torch.cat([x, f_56], dim=1) # 16 + 16 = 32 channels
        # Removed debug print: print(f"After concat3: {x.shape}")
        x = self.conv_block3(x) # 32 -> 16 channels
        # Removed debug print: print(f"After conv_block3: {x.shape}")

        # Decoder Stage 4: Upsample 56x56 -> 112x112
        x = self.upconv4(x) # 16 -> 16 channels, spatial 56x56 -> 112x112
        # Removed debug print: print(f"After upconv4: {x.shape}")
        x = torch.cat([x, f_112], dim=1) # 16 + 16 = 32 channels
        # Removed debug print: print(f"After concat4: {x.shape}")
        x = self.conv_block4(x) # 32 -> 16 channels
        # Removed debug print: print(f"After conv_block4: {x.shape}")

        # Final Upsampling to 224x224
        x = self.upconv_final(x) # 16 channels -> 16 channels, 112x112 -> 224x224
        # Removed debug print: print(f"After final upconv: {x.shape}")

        # Final segmentation head
        output = self.final_conv(x)
        # Removed debug print: print(f"Final output shape: {output.shape}")
        return output

class MobileNetV3UNet(nn.Module):
    def __init__(self, pretrained_encoder=True, freeze_encoder=False, num_classes=1):
        super().__init__()
        self.encoder = MobileNetV3SmallEncoder(pretrained=pretrained_encoder)
        self.decoder = UNetDecoder(self.encoder.out_channels, num_classes)

        if freeze_encoder:
            for param in self.encoder.parameters():
                param.requires_grad = False

    def forward(self, x):
        feature_maps = self.encoder(x)
        output = self.decoder(feature_maps)
        return output

# --- Test the model ---

# Test with pretrained encoder
print("\nTesting MobileNetV3UNet with pretrained encoder:")
model_pretrained = MobileNetV3UNet(pretrained_encoder=True)
model_pretrained.eval() # Set to evaluation mode

# Test with a dummy input
dummy_input = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE)
with torch.no_grad():
    output_pretrained = model_pretrained(dummy_input)

print(f"Output shape (pretrained): {output_pretrained.shape}") # Should be [1, 1, 224, 224]
assert output_pretrained.shape == (1, NUM_CLASSES, IMAGE_SIZE, IMAGE_SIZE), "Pretrained model output shape is incorrect!"
print("Pretrained model test passed.")

# Test with scratch encoder
print("\nTesting MobileNetV3UNet with scratch encoder:")
model_scratch = MobileNetV3UNet(pretrained_encoder=False)
model_scratch.eval()

with torch.no_grad():
    output_scratch = model_scratch(dummy_input)

print(f"Output shape (scratch): {output_scratch.shape}") # Should be [1, 1, 224, 224]
assert output_scratch.shape == (1, NUM_CLASSES, IMAGE_SIZE, IMAGE_SIZE), "Scratch model output shape is incorrect!"
print("Scratch model test passed.")

print("\nMobileNetV3-UNet architecture setup complete.")

## 5. GPU Setup

## 6. Loss Functions and Evaluation Metrics

In [16]:
import torch.nn.functional as F

# --- Loss Functions ---

class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super(DiceLoss, self).__init__()
        self.smooth = smooth

    def forward(self, inputs, targets):
        # Flatten label and prediction tensors
        inputs = inputs.view(-1)
        targets = targets.view(-1)

        intersection = (inputs * targets).sum()
        dice = (2. * intersection + self.smooth) / (inputs.sum() + targets.sum() + self.smooth)

        return 1 - dice

class CombinedLoss(nn.Module):
    def __init__(self, bce_weight=0.5, dice_weight=0.5, smooth=1e-6):
        super(CombinedLoss, self).__init__()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight
        self.dice_loss = DiceLoss(smooth=smooth)
        # BCEWithLogitsLoss combines sigmoid and BCE for numerical stability
        self.bce_loss = nn.BCEWithLogitsLoss()

    def forward(self, inputs, targets):
        bce = self.bce_loss(inputs, targets)
        dice = self.dice_loss(torch.sigmoid(inputs), targets) # Apply sigmoid for Dice Loss as it expects probabilities
        return self.bce_weight * bce + self.dice_weight * dice


# --- Evaluation Metrics ---

def calculate_metrics(predictions, targets, smooth=1e-6):
    # Apply sigmoid to predictions to get probabilities, then threshold to get binary masks
    predictions = torch.sigmoid(predictions)
    predictions = (predictions > 0.5).float()

    # Flatten tensors
    predictions = predictions.view(-1)
    targets = targets.view(-1)

    # True Positives, False Positives, False Negatives, True Negatives
    TP = (predictions * targets).sum() # True Positives
    FP = ((1 - targets) * predictions).sum() # False Positives
    FN = (targets * (1 - predictions)).sum() # False Negatives
    TN = ((1 - targets) * (1 - predictions)).sum() # True Negatives

    # Dice Coefficient
    dice = (2. * TP + smooth) / (2. * TP + FP + FN + smooth)

    # IoU (Jaccard Index)
    iou = (TP + smooth) / (TP + FP + FN + smooth)

    # Precision
    precision = (TP + smooth) / (TP + FP + smooth)

    # Recall (Sensitivity)
    recall = (TP + smooth) / (TP + FN + smooth)

    return dice.item(), iou.item(), precision.item(), recall.item()


# --- Test Loss Functions and Metrics ---

print("\nTesting Loss Functions and Metrics:")

# Create dummy predictions and targets
dummy_preds = torch.randn(BATCH_SIZE, NUM_CLASSES, IMAGE_SIZE, IMAGE_SIZE).to(device)
dummy_targets = torch.randint(0, 2, (BATCH_SIZE, NUM_CLASSES, IMAGE_SIZE, IMAGE_SIZE), dtype=torch.float32).to(device)

# Test Dice Loss
dice_loss_fn = DiceLoss()
dice_loss_output = dice_loss_fn(torch.sigmoid(dummy_preds), dummy_targets)
print(f"Dice Loss (dummy): {dice_loss_output.item():.4f}")

# Test Combined Loss
combined_loss_fn = CombinedLoss()
combined_loss_output = combined_loss_fn(dummy_preds, dummy_targets)
print(f"Combined Loss (dummy): {combined_loss_output.item():.4f}")

# Test Metrics
dice, iou, precision, recall = calculate_metrics(dummy_preds, dummy_targets)
print(f"Metrics (dummy) - Dice: {dice:.4f}, IoU: {iou:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}")

print("Loss functions and evaluation metrics setup complete.")


Testing Loss Functions and Metrics:
Dice Loss (dummy): 0.4996
Combined Loss (dummy): 0.6525
Metrics (dummy) - Dice: 0.4999, IoU: 0.3332, Precision: 0.5003, Recall: 0.4995
Loss functions and evaluation metrics setup complete.


## 7. Training Loop

In [ ]:
import os
import time
import gc
import pandas as pd
import torch
import torch.optim as optim
import shutil
from tqdm import tqdm
from google.colab import drive

# --- 8. Checkpoint directory and Google Drive mount ---
try:
    drive.mount('/content/drive')
except Exception as e:
    print("Google Drive mount skipped or already mounted:", e)

CHECKPOINT_DIR = "/content/drive/MyDrive/ISIC_MobileNetV3_UNet/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# --- 1. Training configuration ---
MAX_EPOCHS = 15
EARLY_STOPPING_PATIENCE = 3
if 'BATCH_SIZE' not in globals():
    BATCH_SIZE = 16

# --- 2. Few-shot sizes ---
FEW_SHOT_SIZES = [50, 100, 250, 500, 1000]

# --- 11. Helper to format seconds ---
def format_time(seconds):
    if seconds < 60:
        return f"{seconds:.1f} sec"
    minutes = seconds / 60
    if minutes < 60:
        return f"{minutes:.1f} min"
    hours = minutes / 60
    return f"{hours:.2f} hr"

# --- 12. T4 GPU verification ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: CUDA is not available. Training will run on CPU.")

# --- Train & Evaluate Orchestration ---
experiment_summaries = []

for fs_size in FEW_SHOT_SIZES:
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        gc.collect()

    # --- 15. Hackathon-friendly output header ---
    print("\n" + "="*50)
    print(f"Starting experiment: {fs_size} images")
    print("="*50)
    print(f"Training images: {fs_size}")
    print(f"Maximum epochs: {MAX_EPOCHS}")
    print(f"Early stopping patience: {EARLY_STOPPING_PATIENCE}")
    print(f"Device: {device}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    print("-"*50)

    # Instantiate architecture
    model = MobileNetV3UNet(pretrained_encoder=True, freeze_encoder=False, num_classes=NUM_CLASSES)
    model = model.to(device)

    # Loss & Optimizer
    criterion = CombinedLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4)

    # --- 8. Learning-rate scheduler ---
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=2
    )

    # --- 6. Validation Dice initialization ---
    best_val_dice = 0.0
    best_val_iou = 0.0
    best_epoch = 0
    epochs_without_improvement = 0

    # --- 13. Timing initialization ---
    training_start_time = time.time()
    completed_epochs = 0

    train_loader = get_train_dataloader(fs_size)

    for epoch in range(MAX_EPOCHS):
        epoch_start_time = time.time()
        completed_epochs = epoch + 1

        # --- Training loop ---
        model.train()
        running_loss = 0.0
        for inputs, masks in train_loader:
            inputs = inputs.to(device)
            masks = masks.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, masks)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * inputs.size(0)

        epoch_train_loss = running_loss / len(train_loader.dataset)

        # --- Validation loop ---
        model.eval()
        val_running_loss = 0.0
        val_dice_total, val_iou_total = 0.0, 0.0

        with torch.no_grad():
            for inputs, masks in validation_loader:
                inputs = inputs.to(device)
                masks = masks.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, masks)
                val_running_loss += loss.item() * inputs.size(0)

                dice, iou, _, _ = calculate_metrics(outputs, masks)
                val_dice_total += dice * inputs.size(0)
                val_iou_total += iou * inputs.size(0)

        epoch_val_loss = val_running_loss / len(validation_dataset)
        val_dice = val_dice_total / len(validation_dataset)
        val_iou = val_iou_total / len(validation_dataset)

        # --- 8. Scheduler Step & LR Print ---
        scheduler.step(val_dice)
        current_lr = optimizer.param_groups[0]["lr"]

        # --- 13 & 14. Epoch Timing & Remaining Time Estimate ---
        epoch_time = time.time() - epoch_start_time
        elapsed = time.time() - training_start_time
        average_epoch_time = elapsed / completed_epochs
        remaining_epochs = MAX_EPOCHS - completed_epochs
        estimated_remaining = average_epoch_time * remaining_epochs

        # --- 15. Hackathon-friendly Output logging ---
        print(f"Epoch {epoch+1}/{MAX_EPOCHS}")
        print(f"Train Loss: {epoch_train_loss:.4f}")
        print(f"Val Loss:   {epoch_val_loss:.4f}")
        print(f"Val Dice:   {val_dice:.4f}")
        print(f"Val IoU:    {val_iou:.4f}")
        print(f"Learning Rate: {current_lr:.6f}")
        print(f"Epoch Time: {epoch_time:.1f} sec")
        print(f"Estimated Remaining: {format_time(estimated_remaining)}")
        print("-"*50)

        # --- 6 & 10. Best validation save ---
        checkpoint_path = os.path.join(CHECKPOINT_DIR, f"mobilenetv3_unet_{fs_size}_best.pth")
        if val_dice > best_val_dice:
            best_val_dice = val_dice
            best_val_iou = val_iou
            best_epoch = epoch + 1
            epochs_without_improvement = 0

            torch.save({
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "best_val_dice": best_val_dice,
                "best_val_iou": best_val_iou,
            }, checkpoint_path)
            print(f"✓ New best validation Dice: {best_val_dice:.4f}")
            print("✓ Checkpoint saved")
        else:
            epochs_without_improvement += 1

        # --- 7. Early Stopping ---
        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print(f"Early stopping triggered.\n")
            print(f"Best Epoch: {best_epoch}")
            print(f"Best Validation Dice: {best_val_dice:.4f}")
            break

    total_train_time = (time.time() - training_start_time) / 60.0
    early_stopped = "Yes" if epochs_without_improvement >= EARLY_STOPPING_PATIENCE else "No"

    # --- 11. Test set isolated evaluation ---
    print(f"\nRestoring best model checkpoint from epoch {best_epoch}...")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])

    model.eval()
    test_dice_total, test_iou_total, test_prec_total, test_rec_total = 0.0, 0.0, 0.0, 0.0

    with torch.no_grad():
        for inputs, masks in test_loader:
            inputs = inputs.to(device)
            masks = masks.to(device)
            outputs = model(inputs)
            dice, iou, precision, recall = calculate_metrics(outputs, masks)
            test_dice_total += dice * inputs.size(0)
            test_iou_total += iou * inputs.size(0)
            test_prec_total += precision * inputs.size(0)
            test_rec_total += recall * inputs.size(0)

    test_dice = test_dice_total / len(test_dataset)
    test_iou = test_iou_total / len(test_dataset)
    test_precision = test_prec_total / len(test_dataset)
    test_recall = test_rec_total / len(test_dataset)

    # --- 16. Final experiment summary ---
    print("\n" + "="*40)
    print(f"Experiment: {fs_size} images")
    print("="*40)
    print(f"Best Epoch: {best_epoch}")
    print(f"Best Validation Dice: {best_val_dice:.4f}")
    print(f"\nTest Dice: {test_dice:.4f}")
    print(f"Test IoU: {test_iou:.4f}")
    print(f"Test Precision: {test_precision:.4f}")
    print(f"Test Recall: {test_recall:.4f}")
    print(f"\nTotal Epochs Completed: {completed_epochs}")
    print(f"Total Training Time: {total_train_time:.1f} minutes")
    print(f"Early Stopping: {early_stopped}")
    print(f"\nCheckpoint: mobilenetv3_unet_{fs_size}_best.pth")
    print("="*40)

    experiment_summaries.append({
        "Training Samples": fs_size,
        "Best Epoch": best_epoch,
        "Best Validation Dice": best_val_dice,
        "Best Validation IoU": best_val_iou,
        "Test Dice": test_dice,
        "Test IoU": test_iou,
        "Test Precision": test_precision,
        "Test Recall": test_recall,
        "Time": f"{total_train_time:.1f} min",
        "Epochs Completed": completed_epochs,
        "Early Stopping": early_stopped,
        "Checkpoint Path": checkpoint_path
    })

# --- 17. Final Results Summary & Save ---
df_results = pd.DataFrame(experiment_summaries)
print("\nTraining Samples | Best Epoch | Test Dice | Test IoU | Time")
print("-"*62)
for idx, row in df_results.iterrows():
    print(f"{row['Training Samples']:<16} | {row['Best Epoch']:<10} | {row['Test Dice']:<9.4f} | {row['Test IoU']:<8.4f} | {row['Time']}")

results_csv_path = os.path.join(CHECKPOINT_DIR, "few_shot_experiment_results.csv")
df_results.to_csv(results_csv_path, index=False)
print(f"\nExperiment CSV saved successfully: {results_csv_path}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda
GPU: Tesla T4

Starting experiment: 50 images
Training images: 50
Maximum epochs: 15
Early stopping patience: 3
Device: cuda
GPU: Tesla T4
--------------------------------------------------
Training Dataset size for 50 images: 50
Epoch 1/15
Train Loss: 0.7258
Val Loss:   0.7105
Val Dice:   0.3401
Val IoU:    0.2154
Learning Rate: 0.000100
Epoch Time: 80.0 sec
Estimated Remaining: 18.7 min
--------------------------------------------------
✓ New best validation Dice: 0.3401
✓ Checkpoint saved
Epoch 2/15
Train Loss: 0.7216
Val Loss:   0.7103
Val Dice:   0.3551
Val IoU:    0.2276
Learning Rate: 0.000100
Epoch Time: 79.3 sec
Estimated Remaining: 17.3 min
--------------------------------------------------
✓ New best validation Dice: 0.3551
✓ Checkpoint saved
Epoch 3/15
Train Loss: 0.7204
Val Loss:   0.7099
Val Dice:   0.3612
Val IoU:    0.2326
Learnin